# Lección 13.3 — GANs: el falsificador y el detective

> **Módulo 13 · Modelos generativos y LLMs** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 9.5 (optimizadores), Lección 13.1 (discriminativo vs. generativo), Lección 13.2 (autoencoders y VAEs)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía cómo una **GAN** aprende enfrentando a dos redes: un **generador** y un **discriminador**.
- Seguir paso a paso cómo el generador mejora **sin ver nunca los datos reales**.
- Leer las curvas de pérdida de una GAN (que **no** bajan como en el aprendizaje supervisado).
- Reconocer el **colapso de modos** y el **desequilibrio** entre las dos redes, y qué perillas los causan.
- Evaluar si unos datos sintéticos se parecen a los reales con una **prueba del detective**.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Prepara los datos de juguete: puntos en el plano que forman un **anillo de 8 manchas**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (8, 4)

# 8 manchas pequeñas colocadas en un círculo: son nuestros "datos reales"
angulos = torch.arange(8) * 2 * np.pi / 8
CENTROS = torch.stack([2 * angulos.cos(), 2 * angulos.sin()], dim=1)

def anillo(n):
    """Devuelve n puntos reales: se elige una mancha al azar y se le suma un poquito de ruido."""
    cual = torch.randint(0, 8, (n,))
    return CENTROS[cual] + 0.05 * torch.randn(n, 2)

def evaluar_anillo(puntos):
    """¿Cuántas de las 8 manchas imita el generador, y qué % de sus puntos cae cerca de alguna?"""
    distancias = torch.cdist(torch.as_tensor(puntos), CENTROS).min(dim=1)
    cerca = distancias.values < 0.2
    por_mancha = torch.bincount(distancias.indices[cerca], minlength=8)
    manchas = int((por_mancha > 0.02 * len(puntos)).sum())   # una mancha "cuenta" si recibe > 2 % de los puntos
    return manchas, cerca.float().mean().item()

torch.manual_seed(0)
reales = anillo(1000)
plt.figure(figsize=(4, 4)); plt.scatter(*reales.T, s=3); plt.title("Datos reales: anillo de 8 manchas")
plt.axis("equal"); plt.show()
print("✅ Todo listo")

---
## 1. 📖 La intuición: el falsificador y el detective

Imagina a dos personajes que empiezan siendo **pésimos** en su trabajo:

- 🎨 **El falsificador** quiere fabricar billetes falsos. Al principio dibuja garabatos.
- 🕵️ **El detective** quiere distinguir billetes reales de falsos. Al principio adivina al azar.

Cada día pasa lo mismo: el detective recibe una mezcla de billetes reales y falsos, y aprende un poco a distinguirlos. Después, el falsificador recibe la **opinión del detective** sobre sus billetes ("este se nota falso por el color") y corrige su técnica.

```
 🎲 ruido ──► 🎨 FALSIFICADOR ──► billete falso ─┐
                                                  ├──► 🕵️ DETECTIVE ──► "¿real o falso?"
                     billetes reales 💵 ──────────┘
           ▲                                                  │
           └──────────── "esto te delató" ◄───────────────────┘
```

Los dos mejoran **compitiendo**. Si todo va bien, al final el falsificador hace billetes tan buenos que el detective solo puede **adivinar al 50 %**.

Esto es una **GAN** (*Generative Adversarial Network*, red generativa antagónica): dos redes neuronales que aprenden jugando una contra la otra.

### ✋ Ejercicio 1 — Calentamiento
1. ¿Qué pasa con el falsificador si el detective es **pésimo** (no distingue nada)?
2. ¿Y si el detective es **perfecto desde el primer día** y solo dice "falso" sin dar pistas de por qué?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. El falsificador **no aprende nada**: cualquier garabato pasa la prueba, así que no tiene motivo para mejorar.
2. Tampoco aprende: si todo lo que hace es "100 % falso" sin pistas, no sabe **hacia dónde** corregir.

El aprendizaje solo funciona si los dos están **más o menos al mismo nivel**: un detective que es un poco mejor que el falsificador, y que le da pistas útiles. Este equilibrio delicado es la mayor dificultad de las GANs (lo verás en la Sección 4).
</details>

---
## 2. 📖 El concepto: dos redes, dos objetivos opuestos

| Pieza | Qué recibe | Qué produce | Qué quiere |
|---|---|---|---|
| **Generador** *G* (falsificador) | Ruido al azar *z* (unos pocos números) | Un ejemplo falso (aquí, un punto; en la vida real, una imagen) | Que *D* diga "real" de sus falsos |
| **Discriminador** *D* (detective) | Un ejemplo, real o falso | Una probabilidad: "¿qué tan real parece?" | Acertar: "real" para los reales, "falso" para los falsos |

El entrenamiento alterna dos pasos, una y otra vez:
1. **Turno del detective:** se le muestran reales y falsos, y se ajusta para clasificarlos mejor (es un clasificador normal, como la regresión logística de la Lección 5.2).
2. **Turno del falsificador:** se generan falsos, se le pregunta al detective, y el generador ajusta sus pesos para que el detective **se equivoque más**. La "pista" viaja hacia atrás a través del detective por retropropagación (Lección 9.4).

> 💡 Igual que el decodificador del VAE (Lección 13.2), el generador convierte **ruido** en **ejemplos**. La diferencia está en **quién lo corrige**: en el VAE, comparar con la imagen original; en la GAN, la opinión de otra red.

### ✋ Ejercicio 2 — ¿Quién ve qué?
1. ¿El generador ve alguna vez un dato real directamente?
2. Al terminar el entrenamiento, ¿cuál de las dos redes se guarda para **crear** contenido nuevo?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Nunca.** Solo recibe ruido y las "pistas" del detective. Aprende cómo son los datos reales **de oídas**, a través de las críticas. Es una de las ideas más sorprendentes del aprendizaje automático.
2. El **generador**. El discriminador fue solo un "entrenador": normalmente se descarta (aunque a veces se reutiliza como detector de falsificaciones).
</details>

---
## 3. 📖 Cómo funciona por dentro

### 3.1 La GAN más pequeña del mundo: un falsificador con 2 perillas
Los datos reales son números alrededor de **4**, con un ancho (desviación estándar) de **0.5**. El falsificador es lo más simple posible: toma ruido *z* y devuelve `media + ancho × z`. Solo tiene **dos perillas**, y empieza con `media = 0` y `ancho = 1`.

El detective es una red pequeña. **Ninguno de los dos conoce el 4 ni el 0.5.**

### 🔮 Ejercicio 3 — Predice
Después de 1,500 rondas del juego:
1. ¿A qué valor crees que llegará la `media` del falsificador?
2. ¿Y el `ancho`? ¿Llegará directo a 0.5 o dará algún rodeo?

✍️ *Mis predicciones:*

In [ ]:
torch.manual_seed(0)
media = torch.zeros(1, requires_grad=True)       # perilla 1 del falsificador
ancho = torch.ones(1, requires_grad=True)        # perilla 2 del falsificador
detective = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 1))
opt_g = torch.optim.Adam([media, ancho], lr=1e-2)
opt_d = torch.optim.Adam(detective.parameters(), lr=1e-2)
bce = nn.BCEWithLogitsLoss()                     # error de clasificación real/falso (entropía cruzada)
uno, cero = torch.ones(128, 1), torch.zeros(128, 1)

for ronda in range(1500):
    reales_1d = 4 + 0.5 * torch.randn(128, 1)            # datos reales (el falsificador no los ve)
    falsos_1d = media + ancho * torch.randn(128, 1)      # lo que fabrica el falsificador
    # Turno del detective: aprender a decir 1 = real, 0 = falso
    perdida_d = bce(detective(reales_1d), uno) + bce(detective(falsos_1d.detach()), cero)
    opt_d.zero_grad(); perdida_d.backward(); opt_d.step()
    # Turno del falsificador: que el detective diga "real" (1) de sus falsos
    perdida_g = bce(detective(falsos_1d), uno)
    opt_g.zero_grad(); perdida_g.backward(); opt_g.step()
    if ronda % 150 == 0 or ronda == 1499:
        print(f"ronda {ronda:>4}: media = {media.item():5.2f} · ancho = {abs(ancho.item()):4.2f} · pérdida del detective = {perdida_d.item():.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. La `media` llega a **≈ 4.0** en unas 300 rondas y se queda ahí.
2. El `ancho` **da un rodeo**: hacia la ronda 300 cae casi a **0** (el falsificador produce **siempre el mismo número**, ≈ 4) y luego se recupera hasta quedar alrededor de **0.4–0.6**, cerca del 0.5 real.

Ese rodeo es una versión en miniatura del **colapso de modos** (Sección 4): al falsificador le convino "copiar lo que mejor engaña" en lugar de imitar toda la variedad. Aquí el detective lo castigó y lo obligó a recuperar la variedad.

Fíjate también en la pérdida del detective: termina en **≈ 1.39** y no baja más. Guarda ese número: lo explicaremos enseguida.
</details>

### 3.2 Ahora una GAN de verdad (en 2D)
Ahora las dos redes son redes neuronales de verdad y los datos, el anillo de 8 manchas. La función de abajo entrena una GAN y guarda "fotos" del generador a lo largo del camino. Fíjate en los comentarios: los dos turnos son los mismos que en la GAN de 2 perillas.

In [ ]:
def entrenar_gan(datos=anillo, pasos=3000, lr_g=1e-3, lr_d=1e-3, dim_ruido=2, semilla=0, fotos_en=()):
    torch.manual_seed(semilla)
    G = nn.Sequential(nn.Linear(dim_ruido, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(),
                      nn.Linear(128, 2)).to(device)                       # falsificador: ruido → punto
    D = nn.Sequential(nn.Linear(2, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(),
                      nn.Linear(128, 1)).to(device)                       # detective: punto → "¿real?"
    opt_g = torch.optim.Adam(G.parameters(), lr=lr_g, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=lr_d, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()
    uno, cero = torch.ones(256, 1, device=device), torch.zeros(256, 1, device=device)
    historial, fotos = {"detective": [], "falsificador": []}, {}
    for paso in range(1, pasos + 1):
        x_real = datos(256).to(device)
        x_falso = G(torch.randn(256, dim_ruido, device=device))
        perdida_d = bce(D(x_real), uno) + bce(D(x_falso.detach()), cero)   # turno del detective
        opt_d.zero_grad(); perdida_d.backward(); opt_d.step()
        perdida_g = bce(D(x_falso), uno)                                   # turno del falsificador
        opt_g.zero_grad(); perdida_g.backward(); opt_g.step()
        historial["detective"].append(perdida_d.item()); historial["falsificador"].append(perdida_g.item())
        if paso in fotos_en:
            fotos[paso] = generar(G, dim_ruido)
    return G, historial, fotos

@torch.no_grad()
def generar(G, dim_ruido=2, n=2000):
    torch.manual_seed(123)                       # siempre el mismo ruido, para comparar con justicia
    return G(torch.randn(n, dim_ruido, device=device)).cpu()

print("✅ Funciones listas")

### 🔮 Ejercicio 4 — La película del entrenamiento
Vamos a entrenar 3,000 pasos (≈ 20–40 s) y tomar fotos en los pasos 1, 250, 1,000 y 3,000.
1. ¿Cómo crees que se verá la primera foto?
2. ¿El generador encontrará las 8 manchas a la vez, o irá "descubriéndolas"?

✍️ *Mis predicciones:*

In [ ]:
FOTOS = (1, 250, 1000, 3000)
G, historial, fotos = entrenar_gan(pasos=3000, fotos_en=FOTOS)

fig, axes = plt.subplots(1, len(FOTOS), figsize=(16, 4))
for ax, paso in zip(axes, FOTOS):
    manchas, calidad = evaluar_anillo(fotos[paso])
    ax.scatter(*reales.T, s=2, c="lightgray")
    ax.scatter(*fotos[paso].T, s=2, c="crimson")
    ax.set_title(f"paso {paso}: {manchas}/8 manchas · {calidad:.0%} cerca", fontsize=10)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
plt.suptitle("Gris = reales · Rojo = falsos del generador"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. En el paso 1 los puntos falsos forman una **nubecita** cerca del centro, lejos de todas las manchas (0/8, 0 % cerca): el generador aún no sabe nada.
2. Ninguna de las dos cosas exactamente: primero la nube **se infla** hasta formar un disco del tamaño del anillo (paso 250: solo ≈ 3/8 manchas y ≈ 15 % de puntos cerca). Hacia el paso 1,000 los puntos ya forman un **polígono** que pasa por las **8 manchas** (≈ 70 % cerca), y en el paso 3,000 la mayoría de los puntos se concentra en ellas (8/8, ≈ 90 % cerca).

Quedan algunos puntos en los "puentes" entre manchas: el generador transforma una nube continua de ruido en algo con 8 islas separadas, y eso le cuesta.
</details>

---
## 4. 📖 ¿Qué pasa si… miro las curvas de pérdida?

En el aprendizaje supervisado (Lección 3.7) la pérdida debería **bajar**. En una GAN las dos pérdidas son **rivales**: si una baja, la otra tiende a subir.

### 🔮 Ejercicio 5 — Predice
Vamos a dibujar las pérdidas de los 3,000 pasos. ¿Bajarán hasta cerca de 0? Si no, ¿en qué valor crees que se estabilizará la del detective? (Pista: la pérdida del detective suma dos errores de clasificación, y si adivina al 50 % cada uno vale ln 2 ≈ 0.69).

✍️ *Mi predicción:*

In [ ]:
def suavizar(v, k=50):
    return np.convolve(v, np.ones(k) / k, mode="valid")

plt.plot(suavizar(historial["detective"]), label="detective (D)")
plt.plot(suavizar(historial["falsificador"]), label="falsificador (G)")
plt.axhline(2 * np.log(2), ls="--", c="gray", label="detective adivinando al 50 % (≈1.39)")
plt.xlabel("paso"); plt.ylabel("pérdida (promedio móvil)"); plt.legend(); plt.grid(alpha=.3)
plt.title("Las pérdidas de una GAN no 'bajan': oscilan"); plt.show()
print(f"Pérdida media del detective en los últimos 500 pasos: {np.mean(historial['detective'][-500:]):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**No bajan a 0.** Las curvas suben y bajan; la del detective, lejos de bajar, **sube** poco a poco y termina en **≈ 1.3–1.4**, muy cerca de 2 × ln 2 ≈ **1.39**: el valor de un detective que **ya no distingue** y adivina al 50 %. Eso es justamente lo que queremos.

Consecuencia práctica muy importante: **en una GAN la pérdida no te dice si las imágenes son buenas.** Hay que **mirar** las muestras o usar métricas aparte (aquí, contar manchas).
</details>

### 📖 Los dos grandes problemas de las GANs

**1. Colapso de modos** (*mode collapse*). El falsificador descubre **un** tipo de falso que engaña bien… y se dedica solo a ese. Es como un falsificador que solo hace billetes de 100 porque son los que mejor le salen. En nuestros datos: un generador que solo produce 2 o 3 de las 8 manchas. Lo viste en miniatura en la GAN de 2 perillas (ancho ≈ 0).

**2. Desequilibrio**. Si una red aprende mucho más rápido que la otra, el juego se rompe:
- Detective demasiado lento → no da pistas útiles → el falsificador persigue un blanco que se mueve sin sentido.
- Detective demasiado rápido → rechaza todo → al falsificador le llegan pistas débiles y avanza muy despacio.

La perilla más directa para el equilibrio es la **tasa de aprendizaje** de cada red.

### 🔮 Ejercicio 6 — Desequilibra el juego
Entrenamos dos GANs más, 1,500 pasos cada una (≈ 30 s en total):
- **A:** detective lento (`lr_d = 1e-4`, diez veces menor que el falsificador).
- **B:** falsificador lento (`lr_g = 1e-4`, diez veces menor que el detective).

¿Cuál crees que fallará más? ¿Alguna sufrirá colapso de modos?

✍️ *Mi predicción:*

In [ ]:
configs = {"A · detective lento": dict(lr_d=1e-4),
           "B · falsificador lento": dict(lr_g=1e-4)}
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, (nombre, cfg) in zip(axes, configs.items()):
    G_x, hist_x, _ = entrenar_gan(pasos=1500, **cfg)
    muestras = generar(G_x)
    manchas, calidad = evaluar_anillo(muestras)
    ax.scatter(*reales.T, s=2, c="lightgray"); ax.scatter(*muestras.T, s=2, c="crimson")
    ax.set_title(f"{nombre}\n{manchas}/8 manchas · {calidad:.0%} cerca", fontsize=10)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
    print(f"{nombre:<24} pérdidas finales → D = {np.mean(hist_x['detective'][-200:]):.2f} · "
          f"G = {np.mean(hist_x['falsificador'][-200:]):.2f} · manchas = {manchas}/8")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **A · detective lento:** fracaso total. Todos los puntos se amontonan en **una sola franja** fuera del anillo: **0/8 manchas**. Es un **colapso de modos** de manual: el detective aprende tan despacio que no le da pistas útiles, y el falsificador se queda "engañándolo" con un único tipo de falso.
- **B · falsificador lento:** va por buen camino pero más despacio: los puntos forman un anillo continuo que pasa por ≈ **6/8 manchas** (≈ 60 % cerca). Con más pasos seguiría mejorando.

Moraleja: en una GAN importa **el equilibrio** entre las dos redes, no solo que cada una sea buena.
</details>

---
## 5. 📖 Hiperparámetros y perillas

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `lr_g` / `lr_d` | Velocidad de aprendizaje de cada red | Aprende más rápido, pero el juego se vuelve inestable | Más estable pero lento; si una es mucho menor, desequilibrio | 1e-4 a 2e-4 en imágenes (aquí 1e-3), **parecidas entre sí** |
| `betas` de Adam | "Memoria" del optimizador (Lección 9.5) | β₁ = 0.9: más inercia, puede oscilar | β₁ = 0.5: reacciona más rápido al rival | `(0.5, 0.999)` |
| `dim_ruido` | Cuántos números de azar recibe el generador | Más variedad posible; más difícil de controlar | Poca variedad → riesgo de colapso | 2–10 en juguete, 100–512 en imágenes |
| `pasos` | Duración del juego | Mejor calidad… hasta que se desestabiliza | Muestras a medio cocinar | Mirar las muestras, no la pérdida |
| Tamaño de las redes | Capacidad de G y D | Formas más complejas | Más rápido | Que D no sea mucho más fuerte que G |
| Tamaño del lote | Cuántos ejemplos ve el detective por turno | Pistas más estables | Más ruido | 64–256 |

### ✏️ Ejercicio 7 — La perilla `dim_ruido`
Entrenamos con `dim_ruido = 1` (el falsificador recibe **un solo** número al azar) y con `dim_ruido = 8`, 1,500 pasos cada uno.

🔮 Antes de ejecutar: con **un solo** número de entrada, el generador dibuja una **línea** (una curva continua) en el plano. ¿Podrá cubrir 8 manchas separadas sin dejar puntos en medio?

✍️ *Mi predicción:*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, d in zip(axes, [1, 8]):                       # ✏️ prueba también 2 o 32
    G_x, _, _ = entrenar_gan(pasos=1500, dim_ruido=d)
    muestras = generar(G_x, dim_ruido=d)
    manchas, calidad = evaluar_anillo(muestras)
    ax.scatter(*reales.T, s=2, c="lightgray"); ax.scatter(*muestras.T, s=2, c="crimson")
    ax.set_title(f"dim_ruido = {d}: {manchas}/8 manchas · {calidad:.0%} cerca", fontsize=10)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **`dim_ruido = 1`:** el generador dibuja literalmente **una línea** que va de mancha en mancha (≈ 7/8 manchas, ≈ 68 % cerca). Como una curva continua no puede "saltar", siempre deja puntos en los tramos entre manchas.
- **`dim_ruido = 8`:** puntos más concentrados en las manchas (≈ 7/8, ≈ 82 % cerca, con los mismos 1,500 pasos).

Con muy poco ruido el generador tiene **poca libertad** para producir variedad. En imágenes reales se usan 100 o más números de ruido.
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Todas las muestras se parecen entre sí | **Colapso de modos** | Igualar tasas de aprendizaje, más `dim_ruido`, variantes como WGAN |
| La pérdida del detective cae a ~0 y el generador no mejora | El detective gana demasiado fácil | Bajar `lr_d`, detective más pequeño, añadir ruido a los reales |
| Las muestras cambian bruscamente de un paso a otro | Juego inestable, tasa de aprendizaje alta | Bajar las dos tasas, `betas=(0.5, 0.999)` |
| "Mi pérdida del generador es la más baja, ¡es el mejor!" | Leer la pérdida como en supervisado | **Mirar las muestras** y usar métricas aparte |
| Las imágenes generadas se parecen demasiado a fotos del entrenamiento | El generador memorizó (riesgo de privacidad) | Buscar el vecino real más cercano de cada muestra |

### 🔮 Ejercicio 8 — No te fíes de la pérdida
Mira los números impresos en el Ejercicio 6 y en el Ejercicio 5. Un compañero solo ve esta tabla y dice: *"La GAN con la pérdida del falsificador más baja es la mejor"*. Según los resultados, ¿tiene razón?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** La GAN con la pérdida del falsificador **más baja** es la **A** (≈ 0.72), ¡y es la peor: 0/8 manchas! La B tiene la pérdida del falsificador **más alta** (≈ 2.2) y cubre 6/8. La GAN principal (Ejercicio 5) termina con ≈ 0.83 y cubre las 8.

La pérdida de cada red depende de **lo bueno que sea su rival**, no solo de ella misma. Por eso, en GANs, **mira las muestras** y usa métricas externas.
</details>

---
## 7. 📋 Ficha del modelo

| | GAN (red generativa antagónica) |
|---|---|
| **Tipo** | Generativo (no necesita etiquetas) |
| **Intuición** | Un falsificador y un detective que mejoran compitiendo |
| **Cómo aprende** | Turnos alternos: D aprende a distinguir real/falso; G aprende a engañar a D usando sus gradientes |
| **Hiperparámetros clave** | `lr_g`, `lr_d`, `dim_ruido`, `betas` de Adam, tamaños de las redes |
| **✅ Fortalezas** | Muestras muy **nítidas** · Generación rapidísima (una sola pasada por G) · Ideas reutilizables (detectar falsos, superresolución) |
| **❌ Debilidades** | Entrenamiento **inestable** · Colapso de modos · La pérdida no indica calidad · Difícil de evaluar |
| **Úsalo cuando…** | Necesitas generar muy rápido (tiempo real) en un dominio acotado (caras, texturas, superresolución) |
| **Evítalo cuando…** | Necesitas variedad y control con texto, o un entrenamiento fácil: hoy se prefiere **difusión** (13.4) |

> 📜 **Un poco de historia:** las GANs (Goodfellow y colaboradores, 2014) dominaron la generación de imágenes durante casi una década. *StyleGAN* produjo las famosas caras de "esta persona no existe". Desde ~2021 los **modelos de difusión** las superaron en calidad y variedad, pero las GANs siguen usándose donde importa la velocidad.

## 8. 🧠 Resumen en 5 puntos
1. Una **GAN** enfrenta a dos redes: el **generador** convierte ruido en ejemplos falsos y el **discriminador** intenta distinguirlos de los reales.
2. El generador **nunca ve datos reales**: aprende solo con las "pistas" (gradientes) del discriminador.
3. Las pérdidas **no bajan**: en el equilibrio, el detective adivina al 50 % (pérdida ≈ 1.39). Para juzgar una GAN hay que **mirar sus muestras**.
4. Sus enemigos son el **colapso de modos** (poca variedad) y el **desequilibrio** entre las redes; las tasas de aprendizaje son la perilla clave.
5. Dan muestras nítidas y rápidas, pero son difíciles de entrenar: por eso la difusión (13.4) tomó el relevo en muchas tareas.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Qué recibe como entrada el generador de una GAN?
- a) Imágenes reales · b) Ruido al azar · c) Etiquetas · d) La salida del discriminador

**2.** Después de entrenar, la pérdida del discriminador se queda cerca de 1.39 (= 2 × ln 2). ¿Qué significa?
- a) El entrenamiento falló · b) El discriminador ya no distingue y adivina al 50 % · c) Sobreajuste · d) Hay que entrenar más hasta llegar a 0

**3.** Tu GAN de caras genera siempre la misma cara con pequeñas variaciones. ¿Cómo se llama el problema?
- a) Sobreajuste · b) Colapso de modos · c) Desvanecimiento del gradiente · d) Fuga de datos

**4.** El discriminador aprende 10 veces más rápido que el generador y su pérdida cae casi a 0. ¿Qué cambio tiene **más** sentido?
- a) Subir `lr_d` · b) Bajar `lr_d` o hacer el discriminador más pequeño · c) Entrenar más pasos sin cambiar nada · d) Quitar el discriminador

**5.** Comparado con un VAE (13.2), una GAN típicamente produce…
- a) Imágenes más borrosas y un entrenamiento más estable · b) Imágenes más nítidas y un entrenamiento más inestable · c) Lo mismo · d) Solo texto

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** El generador transforma ruido en ejemplos.
2. **b)** Es la señal de equilibrio que buscábamos.
3. **b)** Poca variedad = colapso de modos.
4. **b)** Hay que **frenar al detective** para que vuelva a dar pistas útiles.
5. **b)** El VAE promedia (borroso pero estable); la GAN compite (nítida pero inestable).
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Las GANs hicieron posibles los *deepfakes* (videos falsos muy realistas). Si el discriminador de una GAN es justamente un "detector de falsos", ¿por qué no basta con usarlo para detectar *deepfakes*?

**B. Caso de análisis 💼** Un hospital quiere compartir datos de pacientes con una universidad, pero no puede por privacidad. Alguien propone: *"Entrenemos una GAN y compartamos datos sintéticos"*. ¿Qué dos cosas comprobarías antes de aceptar?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

**A.** Porque el generador se entrenó **precisamente para engañar** a ese discriminador: al terminar, el detective adivina casi al 50 %. Además, cada nuevo generador (de otra empresa, con otra arquitectura) deja huellas distintas. Detectar *deepfakes* es una carrera armamentista: se necesitan detectores entrenados con muchos tipos de falsos, marcas de agua y verificación del origen del contenido (Lección 15.5).

**B.** 1) **Utilidad:** que los datos sintéticos se parezcan **de verdad** a los reales (distribuciones, relaciones entre variables, que un modelo entrenado con ellos funcione con datos reales). 2) **Privacidad:** que la GAN **no haya memorizado** pacientes concretos (comparar cada registro sintético con su vecino real más cercano; pacientes raros son los más expuestos). Lo practicarás en la sección siguiente.
</details>

### 9.3 🧩 Práctica integradora: "Datos sintéticos para compartir"

Una empresa tiene datos confidenciales de clientes con **dos variables** (ya escaladas) que forman dos grupos curvos: las famosas "lunas". Quiere compartir con un socio datos **sintéticos** que se parezcan a los reales, pero **sin** revelar ningún cliente concreto.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos reales.**

In [ ]:
lunas_np, _ = make_moons(n_samples=2000, noise=0.05, random_state=0)
LUNAS = torch.tensor(lunas_np, dtype=torch.float32)

def lunas(n):
    """Toma n clientes reales al azar (con repetición)."""
    return LUNAS[torch.randint(0, len(LUNAS), (n,))]

plt.figure(figsize=(5, 3)); plt.scatter(*LUNAS.T, s=2); plt.title("Clientes reales (confidenciales)"); plt.show()

**Paso 2 ▶️ — Entrena dos GANs.** Una "a medio cocinar" (300 pasos) y otra completa (3,000 pasos). Tarda ≈ 30 s.

In [ ]:
G_corta, _, _ = entrenar_gan(datos=lunas, pasos=300)
G_larga, _, _ = entrenar_gan(datos=lunas, pasos=3000)
sinteticos = {"300 pasos": generar(G_corta), "3,000 pasos": generar(G_larga)}

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, (nombre, s) in zip(axes, sinteticos.items()):
    ax.scatter(*LUNAS.T, s=2, c="lightgray"); ax.scatter(*s.T, s=2, c="crimson"); ax.set_title(nombre)
plt.show()

**Paso 3 🔮 — La prueba del detective externo.** Para medir el parecido usamos un **clasificador nuevo e independiente** (*gradient boosting*, Módulo 6) que intenta distinguir reales de sintéticos con validación cruzada. Si los sintéticos fueran perfectos, ¿qué exactitud debería tener? ¿Y si fueran malos?

✍️ *Mi apuesta para cada GAN:*

In [ ]:
for nombre, s in sinteticos.items():
    X_mezcla = np.vstack([LUNAS.numpy(), s.numpy()])
    y_mezcla = np.r_[np.zeros(len(LUNAS)), np.ones(len(s))]      # 0 = real, 1 = sintético
    exactitud = cross_val_score(HistGradientBoostingClassifier(random_state=0), X_mezcla, y_mezcla, cv=5).mean()
    print(f"{nombre:<12} → el detective externo acierta el {exactitud:.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Si los sintéticos fueran perfectos, el detective externo no tendría nada que aprender y acertaría **≈ 50 %** (azar). Si fueran malos, se acercaría al **100 %**.

- GAN de 300 pasos: el detective acierta **≈ 88 %**: los sintéticos se notan (en la figura, ni siquiera tienen la forma de las dos lunas).
- GAN de 3,000 pasos: baja a **≈ 73 %**: mucho mejor, aunque todavía distinguible (le quedan puntos sueltos en los "puentes" entre lunas).

Esta "prueba del detective externo" (*classifier two-sample test*) es una forma práctica y honesta de evaluar datos sintéticos.
</details>

**Paso 4 ✋ — ¿Copió a algún cliente?** Un generador que **memoriza** daría sintéticos casi idénticos a clientes reales. Comparamos la distancia de cada sintético a su cliente real más cercano con la distancia típica **entre dos clientes reales** vecinos.

In [ ]:
from sklearn.neighbors import NearestNeighbors
vecinos = NearestNeighbors(n_neighbors=2).fit(LUNAS.numpy())
entre_reales = vecinos.kneighbors(LUNAS.numpy())[0][:, 1]          # [:, 1]: el vecino más cercano que no es él mismo
sint_a_real = vecinos.kneighbors(sinteticos["3,000 pasos"].numpy(), n_neighbors=1)[0][:, 0]
print(f"Distancia típica entre dos clientes reales vecinos: {np.median(entre_reales):.4f}")
print(f"Distancia típica de un sintético a su real más cercano: {np.median(sint_a_real):.4f}")
print(f"Sintéticos casi idénticos a un cliente real (distancia < 0.001): {(sint_a_real < 0.001).sum()} de {len(sint_a_real)}")
print(f"Como referencia, reales con otro real casi idéntico      : {(entre_reales < 0.001).sum()} de {len(entre_reales)}")

✍️ *¿Qué concluyes?*

<details><summary><b>👀 Ver respuesta</b></summary>

Las dos distancias típicas son prácticamente **iguales** (≈ 0.012): los sintéticos están tan cerca de los reales como los reales entre sí, ni más ni menos. Hay unos pocos sintéticos "casi idénticos" a un cliente (≈ 8 de 2,000), pero **entre los propios reales** hay todavía más pares casi idénticos (≈ 16 de 2,000): es lo esperable cuando hay muchos puntos en un espacio pequeño, no una señal de copia.

Conclusión: **no hay indicios de memorización**. Si la distancia de los sintéticos fuera mucho **menor** que la de los reales entre sí, o si muchos sintéticos coincidieran exactamente con clientes reales, sería una alarma de privacidad.
</details>

**Paso 5 ✋ — Recomendación.** Escribe en 3 frases qué le dirías al gerente: ¿se pueden compartir los sintéticos de la GAN de 3,000 pasos? ¿Qué limitación le advertirías?

✍️ *Mi recomendación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo: *"Los datos sintéticos reproducen bien la forma de los datos reales: un clasificador independiente apenas los distingue, y ningún registro sintético es copia de un cliente real. Recomiendo compartirlos para análisis exploratorios y prototipos. Advertencia: lo comprobamos con 2 variables; con datos reales de muchas columnas hay que repetir estas pruebas (utilidad y privacidad) y revisar especialmente a los clientes raros, que son los más fáciles de memorizar."*

Fíjate en que **no** usamos la pérdida de la GAN para nada: la evaluamos con un **detective externo** y con un chequeo de **privacidad**. Es la lección de la Sección 4.
</details>

---
## 📚 Para profundizar
- 📘 **Artículo original:** Goodfellow et al. (2014), *Generative Adversarial Nets*.
- 🎮 **Interactivo:** *GAN Lab* (poloclub.github.io/ganlab): entrena una GAN en 2D en tu navegador, como la de esta lección.
- 📺 **Video:** StatQuest / Computerphile, "Generative Adversarial Networks" (inglés, con subtítulos).
- 🌐 **Para ver:** thispersondoesnotexist.com (caras generadas con StyleGAN).

---
⬅️ **Anterior:** [13.2 Autoencoders y VAEs: comprimir y reconstruir](13.2-autoencoders-y-vaes.ipynb) · ➡️ **Siguiente:** [13.4 Modelos de difusión: crear imágenes quitando ruido](13.4-modelos-de-difusion.ipynb)